In [2]:
from neo4j import GraphDatabase
from rdflib import Graph, Namespace, Literal, URIRef, RDF
from rdflib.namespace import RDFS, OWL, XSD, FOAF
from pathlib import Path
import re
from tqdm.auto import tqdm

PROJECT_ROOT  = Path.home() / "xai-knowledge-graph"
ONTOLOGY_PATH = PROJECT_ROOT / "ontology" / "xai-kg.ttl"
RDF_DIR       = PROJECT_ROOT / "data" / "rdf"
RDF_DIR.mkdir(parents=True, exist_ok=True)

def load_creds(path=str(Path.home() / "aura_cred.txt")):
    creds = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                creds[k.strip()] = v.strip()
    return creds

creds  = load_creds()
driver = GraphDatabase.driver(creds["NEO4J_URI"], auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"]))
DB     = creds["NEO4J_DATABASE"]

# Set up rdflib graph + namespaces
EX = Namespace("http://xai-kg.example.org/")
DCTERMS = Namespace("http://purl.org/dc/terms/")

g = Graph()
g.bind("",        EX)
g.bind("foaf",    FOAF)
g.bind("dcterms", DCTERMS)
g.bind("owl",     OWL)
g.bind("rdfs",    RDFS)

print("Connected; rdflib graph initialized")

Connected; rdflib graph initialized


## URI helpers:

In [3]:
def slugify(s: str, maxlen: int = 60) -> str:
    """Make a string URI-safe: lowercase, replace non-alphanumerics with _."""
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s[:maxlen] or "unknown"

def paper_uri(arxiv_id):  return EX[f"paper/{arxiv_id}"]
def author_uri(name):     return EX[f"author/{slugify(name)}"]
def venue_uri(name):      return EX[f"venue/{slugify(name)}"]
def topic_uri(name):      return EX[f"topic/{slugify(name)}"]

In [4]:
with driver.session(database=DB) as s:
    # Papers
    print("Exporting Papers...")
    papers = list(s.run("MATCH (p:Paper) RETURN p"))
    for record in tqdm(papers, desc="Papers"):
        p = record["p"]
        u = paper_uri(p["arxiv_id"])
        g.add((u, RDF.type, EX.Paper))
        g.add((u, EX.arxivId, Literal(p["arxiv_id"])))
        if p.get("title"):           g.add((u, EX.title,         Literal(p["title"])))
        if p.get("abstract"):        g.add((u, EX["abstract"],   Literal(p["abstract"])))
        if p.get("year") is not None: g.add((u, EX.year,          Literal(p["year"], datatype=XSD.integer)))
        if p.get("citation_count") is not None:
            g.add((u, EX.citationCount, Literal(p["citation_count"], datatype=XSD.integer)))

    # Authors
    print("Exporting Authors...")
    authors = list(s.run("MATCH (a:Author) RETURN a"))
    for record in tqdm(authors, desc="Authors"):
        a = record["a"]
        u = author_uri(a["name"])
        g.add((u, RDF.type, EX.Author))
        g.add((u, EX.name, Literal(a["name"])))

    # Venues
    print("Exporting Venues...")
    venues = list(s.run("MATCH (v:Venue) RETURN v"))
    for record in tqdm(venues, desc="Venues"):
        v = record["v"]
        u = venue_uri(v["name"])
        g.add((u, RDF.type, EX.Venue))
        g.add((u, EX.name, Literal(v["name"])))

    # Topics
    print("Exporting Topics...")
    topics = list(s.run("MATCH (t:Topic) RETURN t"))
    for record in tqdm(topics, desc="Topics"):
        t = record["t"]
        u = topic_uri(t["name"])
        g.add((u, RDF.type, EX.Topic))
        g.add((u, EX.name, Literal(t["name"])))

print(f"\nTriples after node export: {len(g):,}")

Exporting Papers...


Papers:   0%|          | 0/3907 [00:00<?, ?it/s]

Exporting Authors...


Authors:   0%|          | 0/13933 [00:00<?, ?it/s]

Exporting Venues...


Venues:   0%|          | 0/733 [00:00<?, ?it/s]

Exporting Topics...


Topics:   0%|          | 0/30 [00:00<?, ?it/s]


Triples after node export: 52,827


In [5]:
with driver.session(database=DB) as s:
    # AUTHORED
    print("Exporting AUTHORED...")
    rels = list(s.run("MATCH (a:Author)-[:AUTHORED]->(p:Paper) RETURN a.name AS author, p.arxiv_id AS paper"))
    for r in tqdm(rels, desc="AUTHORED"):
        g.add((author_uri(r["author"]), EX.authored, paper_uri(r["paper"])))

    # CITES
    print("Exporting CITES...")
    rels = list(s.run("MATCH (a:Paper)-[:CITES]->(b:Paper) RETURN a.arxiv_id AS citing, b.arxiv_id AS cited"))
    for r in tqdm(rels, desc="CITES"):
        g.add((paper_uri(r["citing"]), EX.cites, paper_uri(r["cited"])))

    # PUBLISHED_IN
    print("Exporting PUBLISHED_IN...")
    rels = list(s.run("MATCH (p:Paper)-[:PUBLISHED_IN]->(v:Venue) RETURN p.arxiv_id AS paper, v.name AS venue"))
    for r in tqdm(rels, desc="PUBLISHED_IN"):
        g.add((paper_uri(r["paper"]), EX.publishedIn, venue_uri(r["venue"])))

    # ABOUT
    print("Exporting ABOUT...")
    rels = list(s.run("MATCH (p:Paper)-[:ABOUT]->(t:Topic) RETURN p.arxiv_id AS paper, t.name AS topic"))
    for r in tqdm(rels, desc="ABOUT"):
        g.add((paper_uri(r["paper"]), EX.about, topic_uri(r["topic"])))

print(f"\nTriples after relationship export: {len(g):,}")

Exporting AUTHORED...


AUTHORED:   0%|          | 0/17119 [00:00<?, ?it/s]

Exporting CITES...


CITES:   0%|          | 0/2637 [00:00<?, ?it/s]

Exporting PUBLISHED_IN...


PUBLISHED_IN:   0%|          | 0/3299 [00:00<?, ?it/s]

Exporting ABOUT...


ABOUT:   0%|          | 0/11396 [00:00<?, ?it/s]


Triples after relationship export: 87,278


In [6]:
output_path = RDF_DIR / "xai_kg_data.ttl"
g.serialize(destination=str(output_path), format="turtle")
print(f"Saved {len(g):,} triples to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

Saved 87,278 triples to /slurm/homes/pchandra/xai-knowledge-graph/data/rdf/xai_kg_data.ttl
File size: 9.2 MB
